In [1]:
import pyexasol
import configparser
import pandas as pd
#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\svi02\\.spyder-py3\\ExasolPROD.ini')

['C:\\Users\\svi02\\.spyder-py3\\ExasolPROD.ini']

In [2]:
dsn=config['exasolPROD']['dsn']
user=config['exasolPROD']['user']
pwd=config['exasolPROD']['pwd']
schema=config['exasolPROD']['schema']

In [3]:
# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)

In [4]:
# destination_analysis_list = ['25_ABE08_20190731_142851',
#                         '120626_KIM01_20180918_094951',
#                         '25_ABE08_20190801_180133',
#                         '119766_ABE08_20190807_180720',
#                         '119766_ABE08_20190805_151438'
#                         ]
destination_analysis_df = pd.read_excel("C:\\Users\\svi02\\Documents\\CSA_BUG\\csa_insert_2019_07.xlsx" ,
                                           sheet_name = 'Grid Results',
                                           header=0)
destination_analysis_list = destination_analysis_df.values.tolist()
df = pd.DataFrame()
a=[]

In [5]:
destination_analysis_list[0:5]

[['118992_ABE08_20190701_144538', '118992_ABE08_20190701_144538'],
 ['488_SSH17_20190702_100034', '488_SSH17_20190702_100034'],
 ['5609_AKU07_20190703_095700', '5609_AKU07_20190703_095700'],
 ['4375_SGO02_20190703_201827', '4375_SGO02_20190703_201827'],
 ['8825_CPE06_20190705_135233', '8825_CPE06_20190705_135233']]

In [36]:
#file_read
destination_analysis = '118992_ABE08_20190701_144538'
import_old_script = 'C:\\Users\\svi02\\Documents\\CSA_BUG\\OLD_SQL_AUTO_test.txt'
script_open = open(import_old_script, 'r')
script_read = script_open.read()  
script_read = script_read.replace('xxxx', "'" +str(destination_analysis) +"'")
#print(script_read)
QUERY = connect.execute(script_read)
#print(QUERY)
# Importing data into a DataFrame
for row in QUERY:

    a.append(row)
    #print(a)
df = pd.DataFrame(a)
df_col_names = QUERY.col_names
df.columns = df_col_names
print(df)   

     HOTEL_ID                               HOTEL_NAME  \
0       23124               ACHAT Comfort Schwetzingen   
1       15050                           Courtyard Linz   
2      214174                                Killashee   
3      171420           Austria Trend Hotel Messe Wien   
4      370612         Hilton Virginia Beach Oceanfront   
..        ...                                      ...   
531    179197      MARIAZELLERHOF PENSION-APPARTEMENTS   
532    939357                         Hotel Las Villas   
533     90198                 Hotel Barbarossa Classic   
534    136344  Holiday Inn Express INDIANAPOLIS SOUTH    
535    619947                      Fraser Suites Perth   

                    ANALYSIS_NAME  \
0    118992_ABE08_20190701_144538   
1    118992_ABE08_20190701_144538   
2    118992_ABE08_20190701_144538   
3    118992_ABE08_20190701_144538   
4    118992_ABE08_20190701_144538   
..                            ...   
531  118992_ABE08_20190701_144538   
532  11

In [32]:
for destination_analysis in destination_analysis_list[0]:

#file_read
    import_old_script = 'C:\\Users\\svi02\\Documents\\CSA_BUG\\OLD_SQL_AUTO_test.txt'
    script_open = open(import_old_script, 'r')
    script_read = script_open.read()  
    script_read = script_read.replace('xxxx', str(destination_analysis).replace('[', '').replace(']',''))
    #print(script_read)
    QUERY = connect.execute(script_read)
    #print(QUERY)
# Importing data into a DataFrame
    for row in QUERY:
       
        a.append(row)
        #print(a)
    df = pd.DataFrame(a)
    df_col_names = QUERY.col_names
    df.columns = df_col_names
print(df)    
# df.to_excel('C:\\Users\\svi02\\Documents\\CSA_BUG\\OLD_SQL_CSA_EXPORT.xlsx', 
#             sheet_name='old_csa_export',
#             index=False)

ExaQueryError: 
(
    message     =>  syntax error, unexpected $undefined, expecting ')' [line 16, column 129] (Session: 1642488023350086336)
    dsn         =>  10.146.96.211..215:8563
    user        =>  app_bus07
    schema      =>  DWHBIL
    code        =>  42000
    session_id  =>  1642488023350086336
    query       =>  WITH "#EXCHANGE_RATE" AS ( -- Report Currency
	SELECT
		CURRENCY_EXCHANGE_RATE
	FROM
		DWHBIL.V_LKP_CURRENCY_EXCHANGE_RATE
	WHERE
		CURRENCY_ISO = 'EUR' -- [F_Currency Iso =?!]
) ,  "#HOTEL_RATING" AS ( -- Hotel Rating
	SELECT
		  HOTEL_ID
		, ROUND(SUM(HOTEL_RATING) / COUNT(DISTINCT HOTEL_RATING_ID), 2) AS HOTEL_RATING
	FROM
		DWHBIL.V_LKP_HOTEL_RATING
	WHERE
		IS_ACTIVE_RATING = 1 AND RATING_CREATION_DATE  >= curdate() - 730
			AND HOTEL_ID IN (SELECT DISTINCT HOTEL_ID FROM DWHBIL.FAK_FOCUS_DESTINATION_ANALYSIS WHERE DESTINATION_ANALYSIS_NAME = 118992_ABE08_20190701_144538
			AND HOTEL_ID > 0)
	GROUP BY
		HOTEL_ID
) ,  "#AUDIT_RESULTS_CLIENT" AS ( -- Rate Audit

In [7]:
for destination_analysis in destination_analysis_list:
#file_read
    import_new_script = 'C:\\Users\\svi02\\Documents\\CSA_BUG\\NEW_SQL_AUTO.txt'
    script_open = open(import_new_script, 'r')
    script_read = script_open.read()  
    script_read = script_read.replace('xxxx', str(destination_analysis).replace('[', '').replace(']',''))
    #print(script_read)
    QUERY = connect.execute(script_read)
# Importing data into a DataFrame
  
    for row in QUERY:
       
        a.append(row)
    print(len(a))
    df = pd.DataFrame(a)
    df_col_names = QUERY.col_names
    df.columns = df_col_names
df.to_excel('C:\\Users\\svi02\\Documents\\CSA_BUG\\NEW_SQL_CSA_EXPORT.xlsx', 
            sheet_name='new_csa_export',
            index=False)

207
208
209
210
211
212
212
212
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
255
255
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
278
279
280
281
282
283
284
285
286
287
287
288
288
289
290
291
292
293
294
295
295
295
295
296
297
298
299
300
301
302
303
304
305
306
307
308
309
310
311
312
313
314
315
316
317
318
319
320
321
322
323
324
325
326
327
328
329
330
331
332
333
334
335
336
336
336
336
337
338
339
340
340
341
342
343
344
345
346
347
348
349
350
351
352
353
354
355
356
357
358
359
360
361
362
363
364
365
366
367
368
369
370
371
372
373
374
375
376
377
377
377
377
378
379
380
381
381
382
383
384
385
386
387
388
389
390
391
392
393
393
394
395
396
397
398
399
400
401
402
403
404
405
406
407
408
409
410
411
412
